In [ ]:
from src.components.config import OUTPUT_UNITS, SEQUENCE_LENGTH
from src.components.preprocess import run_preprocessing_pipeline
from src.components.train import train_model

EPOCHS = 100
BATCH_SIZE = 64
SEQUENCE_STRIDE = 1

In [ ]:
# Run preprocessing once before training to refresh dataset and mappings.
summary = run_preprocessing_pipeline()
summary

In [ ]:
_, history = train_model(
    output_units=OUTPUT_UNITS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    sequence_length=SEQUENCE_LENGTH,
    sequence_stride=SEQUENCE_STRIDE,
    validation_size=0.1,
    model_path="main/model.h5",
)

history.keys()

In [ ]:
history["val_loss"][-1], history["val_accuracy"][-1]

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, None, 45)]        0         
                                                                 
 lstm_1 (LSTM)               (None, 256)               309248    
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                                 
 dense_1 (Dense)             (None, 45)                11565     
                                                                 
Total params: 320813 (1.22 MB)
Trainable params: 320813 (1.22 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/100
67/67 [==============================] - 9s 126ms/step - loss: 1.2993 - accuracy: 0.7536
Epoch 2/100
67/67 [==============================] - 8

c:\Users\udayp\anaconda3\envs\MLenv\lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
from src.components.generate import MelodyGenerator

In [ ]:
generator = MelodyGenerator(model_path="main/model.h5", mapping_path="main/mapping.json")
seed = "67 _ 67 _ 67 _ _ 65 64 _ 64 _ 64 _ _"
melody = generator.generate_melody(seed=seed, num_steps=500, max_sequence_length=SEQUENCE_LENGTH, temperature=0.3)
generator.save_melody(melody, output_path="main/mel.mid")
melody[:50]

1/1 [==============================] - 0s 17ms/step
['67', '_', '67', '_', '67', '_', '_', '65', '64', '_', '64', '_', '64', '_', '_', '_', 'r', '_', '_', '_', '67', '_', '_', '_', '64', '_', '_', '_', '67', '_', '_', '_', '65', '_', '_', '_', '65', '_', '_', '_', '65', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '62', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '65', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '65', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '65', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '65', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '_', '_', '65', '_', '64', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '64', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_', '_'

In [1]:
import os
import random
from django.shortcuts import render
from django.http import FileResponse
from django import forms

# Dummy function to simulate AI model generating a MIDI file
def generate_midi(notes):
    midi_path = "generated_midi.mid"
    with open(midi_path, "wb") as f:
        f.write(os.urandom(100))  # Replace with actual MIDI generation logic
    return midi_path

class NoteForm(forms.Form):
    notes = forms.CharField(label='Enter numbers (21-108) separated by commas', max_length=255)

def generate_midi_view(request):
    if request.method == "POST":
        form = NoteForm(request.POST)
        if form.is_valid():
            notes_str = form.cleaned_data['notes']
            try:
                notes = [int(n) for n in notes_str.split(',') if 21 <= int(n) <= 108]
                if not notes:
                    raise ValueError("Invalid range")
                midi_file = generate_midi(notes)
                return FileResponse(open(midi_file, "rb"), as_attachment=True, filename="generated_midi.mid")
            except ValueError:
                form.add_error('notes', "Please enter valid numbers between 21 and 108.")
    else:
        form = NoteForm()
    return render(request, "generate_midi.html", {"form": form})
